# BEAD Random Forest — Best AIC Features (Log-Transformed)
Predicts `log(funding_per_location)` using the 8 features selected by AIC:
technology, avg_latency, total_fiber_miles, miles_per_location, priority_broadband_project, state_num_providers, state_population, incumbent_democrat

Dropped by AIC: `jobs_per_location`, `project_type_code`

In [ ]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

client = bigquery.Client(project='broadband-data')
print('Connected to BigQuery: broadband-data')

In [ ]:
project_query = """
SELECT project_id, state, bead_support,
       estimated_miles_aerial_fiber, estimated_miles_buried_fiber,
       estimated_jobs, project_type, priority_broadband_project
FROM `broadband-data.fp_approved.deployment_projects`
"""
df_projects = client.query(project_query).to_dataframe()
print(f'Projects: {df_projects.shape}')

In [ ]:
location_query = """
SELECT project_id, COUNT(*) AS funded_locations,
       SAFE_CAST(APPROX_TOP_COUNT(CAST(technology AS STRING), 1)[OFFSET(0)].value AS FLOAT64) AS technology,
       AVG(CAST(low_latency AS INT64)) AS avg_latency
FROM `broadband-data.fp_approved.locations`
GROUP BY project_id
"""
df_locations = client.query(location_query).to_dataframe()
print(f'Location aggregates: {df_locations.shape}')

In [ ]:
state_name_to_abbr = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA',
    'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE', 'Florida': 'FL', 'Georgia': 'GA',
    'Hawaii': 'HI', 'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA',
    'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV', 'New_Hampshire': 'NH',
    'New_Jersey': 'NJ', 'New_Mexico': 'NM', 'New_York': 'NY', 'North_Carolina': 'NC',
    'North_Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA',
    'Rhode_Island': 'RI', 'South_Carolina': 'SC', 'South_Dakota': 'SD', 'Tennessee': 'TN',
    'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA',
    'West_Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY', 'District_of_Columbia': 'DC'
}
nbm_query = """
SELECT state, COUNT(DISTINCT frn) AS state_num_providers
FROM `broadband-data.fcc_nbm.nbm_hive`
GROUP BY state
"""
df_nbm = client.query(nbm_query).to_dataframe()
df_nbm['state'] = df_nbm['state'].map(state_name_to_abbr)
df_nbm = df_nbm.dropna(subset=['state'])
print(f'State provider counts: {df_nbm.shape}')

In [ ]:
state_pop_query = """
SELECT stateabbr AS state, SUM(pop2020) AS state_population
FROM `broadband-data.fcc_block_level_pop.us2020`
GROUP BY stateabbr
"""
df_state_pop = client.query(state_pop_query).to_dataframe()
print(f'State population: {df_state_pop.shape}')

In [ ]:
governor_party_2024 = {
    'AL': 0, 'AK': 0, 'AZ': 1, 'AR': 0, 'CA': 1,
    'CO': 1, 'CT': 1, 'DE': 1, 'FL': 0, 'GA': 0,
    'HI': 1, 'ID': 0, 'IL': 1, 'IN': 0, 'IA': 0,
    'KS': 1, 'KY': 1, 'LA': 0, 'ME': 1, 'MD': 1,
    'MA': 1, 'MI': 1, 'MN': 1, 'MS': 0, 'MO': 0,
    'MT': 0, 'NE': 0, 'NV': 0, 'NH': 0, 'NJ': 1,
    'NM': 1, 'NY': 1, 'NC': 1, 'ND': 0, 'OH': 0,
    'OK': 0, 'OR': 1, 'PA': 1, 'RI': 1, 'SC': 0,
    'SD': 0, 'TN': 0, 'TX': 0, 'UT': 0, 'VT': 0,
    'VA': 0, 'WA': 1, 'WV': 0, 'WI': 1, 'WY': 0, 'DC': 1
}
df_gov = pd.DataFrame(list(governor_party_2024.items()), columns=['state', 'incumbent_democrat'])
print(f'Governor party: D={df_gov["incumbent_democrat"].sum()}, R={(df_gov["incumbent_democrat"]==0).sum()}')

In [ ]:
# Merge all sources
df = df_projects.merge(df_locations, on='project_id', how='left')
df = df.merge(df_nbm, on='state', how='left')
df = df.merge(df_state_pop, on='state', how='left')
df = df.merge(df_gov, on='state', how='left')

df['funded_locations'] = df['funded_locations'].fillna(0)
df['technology'] = df['technology'].fillna(0)
df['avg_latency'] = df['avg_latency'].fillna(0)
df['state_num_providers'] = df['state_num_providers'].fillna(0)
df['state_population'] = df['state_population'].fillna(0)
df['incumbent_democrat'] = df['incumbent_democrat'].fillna(0).astype(int)
print(f'Merged: {df.shape}')

In [ ]:
# Feature engineering + target
df['total_fiber_miles'] = df['estimated_miles_aerial_fiber'].fillna(0) + df['estimated_miles_buried_fiber'].fillna(0)
df['miles_per_location'] = df['total_fiber_miles'] / df['funded_locations'].replace(0, np.nan)
df['miles_per_location'] = df['miles_per_location'].fillna(0)
df['priority_broadband_project'] = df['priority_broadband_project'].map({True: 1, False: 0, 'Yes': 1, 'No': 0, 1: 1, 0: 0}).fillna(0).astype(int)

df['funding_per_location'] = df['bead_support'] / df['funded_locations'].replace(0, np.nan)
df = df.dropna(subset=['funding_per_location'])

low = df['funding_per_location'].quantile(0.025)
high = df['funding_per_location'].quantile(0.975)
df = df[(df['funding_per_location'] >= low) & (df['funding_per_location'] <= high)]

# Log-transform target
df['log_funding_per_location'] = np.log1p(df['funding_per_location'])
print(f'After outlier removal: {df.shape[0]} projects')
print(df['log_funding_per_location'].describe())

In [ ]:
# AIC-selected features (8 of 10 — dropped jobs_per_location, project_type_code)
feature_cols = [
    'technology',
    'avg_latency',
    'total_fiber_miles',
    'miles_per_location',
    'priority_broadband_project',
    'state_num_providers',
    'state_population',
    'incumbent_democrat',
]

X = df[feature_cols].fillna(0)
y = df['log_funding_per_location']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

rf = RandomForestRegressor(n_estimators=200, min_samples_split=5, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

# Metrics in log space
r2 = r2_score(y_test, y_pred)
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_r2 = cross_val_score(rf, X, y, cv=cv, scoring='r2')

# Metrics in real $ space
y_test_real = np.expm1(y_test)
y_pred_real = np.expm1(y_pred)
rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))
mae = mean_absolute_error(y_test_real, y_pred_real)

print(f'Features: {len(feature_cols)} | Samples: {len(X)}')
print(f'\nTest R² (log space):   {r2:.3f}')
print(f'5-fold CV R² (log):    {cv_r2.mean():.3f} ± {cv_r2.std():.3f}')
print(f'RMSE (real $):         {rmse:,.0f}')
print(f'MAE (real $):          {mae:,.0f}')

In [ ]:
# Predicted vs Actual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred, alpha=0.4, s=20, color='steelblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--')
axes[0].set_xlabel('Actual log(Funding per Location)')
axes[0].set_ylabel('Predicted')
axes[0].set_title(f'Log Scale  |  R²={r2:.3f}')

axes[1].scatter(y_test_real, y_pred_real, alpha=0.4, s=20, color='steelblue')
lims2 = [min(y_test_real.min(), y_pred_real.min()), max(y_test_real.max(), y_pred_real.max())]
axes[1].plot(lims2, lims2, 'r--')
axes[1].set_xlabel('Actual Funding per Location ($)')
axes[1].set_ylabel('Predicted')
axes[1].set_title('Real $ Scale')

plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 5))
plt.barh(importance_df['feature'][::-1], importance_df['importance'][::-1], color='steelblue')
plt.xlabel('Importance')
plt.title('Feature Importances — AIC-Best Log RF')
plt.tight_layout()
plt.show()

print(importance_df.to_string(index=False))